# NB05 — External validation on eICU (overall)

**Purpose.** Apply the frozen MIMIC models to all eICU patients (zero adaptation). Report the full calibration hierarchy with hospital-clustered bootstrap CIs, and compare with APACHE IVa on the same patients.

**Research question.** RQ1 — how does a MIMIC-trained model transport overall?

| | |
|---|---|
| **Inputs** | `models/*_V1.pkl`, analysis parquet |
| **Outputs** | `models/pred_eicu.parquet`, Table 2, Figure 2 |
| **Free-Colab runtime** | 10–20 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB05')

In [ ]:
import joblib
m, e = tu.load_analysis()
FEATS = tu.feature_set('V1')
models = {'LightGBM': 'lgbm_V1.pkl', 'Logistic regression': 'lr_V1.pkl', 'XGBoost': 'xgb_V1.pkl'}
pcol = {'LightGBM': 'p_lgbm', 'Logistic regression': 'p_lr', 'XGBoost': 'p_xgb'}
Xe = e[FEATS].astype(float)
pred = e[['stay_id', 'hospital_id', 'year', 'split', 'random_split', 'mortality', 'apache_pred', 'age',
          'sex_male', 'race_group']].copy()
for k, f in models.items():
    pred[pcol[k]] = joblib.load(os.path.join(tu.MODELS, f)).predict_proba(Xe)[:, 1]
pred.to_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'), index=False)
pm = pd.read_parquet(os.path.join(tu.MODELS, 'pred_mimic.parquet'))
mt = pm[pm['set'] == 'temporal_test']
print('eICU predictions saved:', pred.shape)

In [ ]:
B = tu.n_boot(500)   # 500 percentile-bootstrap replicates (each replicate re-scores ~50-95k patients)
rows = []
def _boot(label, dataset, y, p, clusters):
    ci, _ = tu.bootstrap(y, p, clusters=clusters, B=B, label=f'{label} / {dataset}')
    return ci
def add(label, dataset, y, p, clusters=None):
    # cached per comparison: after a Colab disconnect, finished comparisons are reloaded, not recomputed
    key = ('boot_' + label + '_' + dataset).replace(' ', '_').replace('(', '').replace(')', '')
    ci = tu.cached(key, _boot, label, dataset, y, p, clusters)
    for _, r in ci.iterrows():
        rows.append(dict(model=label, dataset=dataset, metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi']))
    print(f'done: {label} / {dataset}', flush=True)

for k in models:
    add(k, 'MIMIC temporal test', mt['mortality'].values, mt[pcol[k]].values)
    add(k, 'eICU all (zero adaptation)', pred['mortality'].values, pred[pcol[k]].values, pred['hospital_id'].values)
sub = pred[pred['apache_pred'].notna()]
add('LightGBM', 'eICU APACHE-available subset', sub['mortality'].values, sub['p_lgbm'].values, sub['hospital_id'].values)
add('APACHE IVa', 'eICU APACHE-available subset', sub['mortality'].values, sub['apache_pred'].values, sub['hospital_id'].values)
t2 = pd.DataFrame(rows)
tu.save_table(t2.round(4), 'table2_long', show=False)
t2w = t2.assign(txt=t2.apply(lambda r: tu.fmt_ci(r.estimate, r.lo, r.hi), axis=1)) \
        .pivot_table(index=['dataset', 'model'], columns='metric', values='txt', aggfunc='first')
tu.save_table(t2w[['auroc', 'auprc', 'scaled_brier', 'oe', 'citl', 'slope', 'ici', 'ece10']], 'table2_transport', index=True)

In [ ]:
# Paired, hospital-clustered comparison of LightGBM vs APACHE IVa on identical patients
cmp = [tu.paired_bootstrap_diff(sub['mortality'].values, sub['p_lgbm'].values, sub['apache_pred'].values,
                                metric=mt_, clusters=sub['hospital_id'].values, B=tu.n_boot(500))
       for mt_ in ['auroc', 'brier', 'scaled_brier']]
cmp = pd.DataFrame(cmp)
tu.save_table(cmp.round(4), 'lgbm_vs_apache_paired')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
tu.plot_calibration(axes[0], {'MIMIC temporal test': (mt['mortality'].values, mt['p_lgbm'].values),
                              'eICU (zero adaptation)': (pred['mortality'].values, pred['p_lgbm'].values)},
                    'A. LightGBM', hist_p=pred['p_lgbm'].values)
tu.plot_calibration(axes[1], {'MIMIC temporal test': (mt['mortality'].values, mt['p_lr'].values),
                              'eICU (zero adaptation)': (pred['mortality'].values, pred['p_lr'].values)},
                    'B. Logistic regression', hist_p=pred['p_lr'].values)
tu.plot_calibration(axes[2], {'LightGBM': (sub['mortality'].values, sub['p_lgbm'].values),
                              'APACHE IVa': (sub['mortality'].values, sub['apache_pred'].values)},
                    'C. eICU: LightGBM vs APACHE IVa', hist_p=sub['apache_pred'].values)
tu.save_fig(fig, 'fig2_calibration_transport'); plt.show()

In [ ]:
tu.finish_notebook({
    'n_eicu': len(pred), 'n_apache_subset': len(sub),
    'table2': t2.round(4).to_dict('records'),
    'lgbm_vs_apache': cmp.round(4).to_dict('records'),
})